# Video-LLaVA / MuLER Stage 1 Pretraining on Google Colab (Hybrid Drive + Fast SSD)

### Architecture & Pipeline Overview
Stage 1 pretrains the multimodal projector on both **images (LLaVA-558K)** and **videos (Valley)**.
To maximize A100 GPU throughput while staying within Colab's storage limits, we use a **hybrid Drive + NVMe SSD architecture**:

| Component | Location | Mechanism & Benefit |
|-----------|----------|---------------------|
| **Video dataset** (Valley, 463 GB) | **Google Drive** | Streamed directly from persistent Drive (`datasets/valley/`). Way too large for local SSD. |
| **Image dataset** (558K images) | **Google Drive → Local SSD** | Stored on Drive as `llava_image.zip` (26.5 GB); auto-unpacked to local NVMe SSD in 45s on first launch. Feeds A100 at 2 GB/s with 0 network latency. |
| **Annotation JSONs** | **Local SSD** | Synced and auto-normalized from Drive to local SSD (`/content/data/pt_json`). Fast reads. |
| **Model cache** | **Local SSD** | Fast loading into GPU VRAM. |
| **Checkpoints** | **Local SSD → Drive** | Fast local checkpoint writes, auto-muled asynchronously to Google Drive every 20s. |

### Datasets Used:
- **Image Pretrain**: LLaVA-558K (`llava_image.zip` from HuggingFace)
- **Video Pretrain**: Valley Video (`valley_2.zip.001`-`012` from HuggingFace)
- **Annotations**: Official zip from Video-LLaVA authors

## 1. Check GPU Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

> **Note for Omar (Shared Folder)**:
> If accessing the dataset from a shared Google Drive folder:
> 1. Go to Google Drive web (`drive.google.com`) -> **Shared with me**.
> 2. Right-click the **Video-LLaVA** folder -> **Add shortcut to Drive** -> select **My Drive**.
> 3. Run the cell below to mount and verify.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Video-LLaVA"
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"Google Drive workspace ready at: {DRIVE_ROOT}")

# Verification Check
valley_path = f"{DRIVE_ROOT}/datasets/valley"
image_zip = f"{DRIVE_ROOT}/datasets/llava_image.zip"
image_folder = f"{DRIVE_ROOT}/datasets/llava_image"

v_ok = os.path.exists(valley_path)
i_ok = os.path.exists(image_zip) or os.path.exists(image_folder)

if v_ok and i_ok:
    print("\u2705 Datasets verified on Google Drive! Ready for pretraining.")
    if os.path.exists(image_zip):
        print(f"   \u2022 Image Master Archive: {image_zip} ({os.path.getsize(image_zip)/(1024**3):.2f} GB) [45s fast SSD staging enabled]")
else:
    print("\u26a0\ufe0f Note: Datasets not found at default path. If using a shared folder, ensure 'Add shortcut to Drive' was performed in MyDrive.")

### 2B. One-Time Setup: Download Image Archive & Normalize Video Paths
> **Note**: Only run this cell if `llava_image.zip` is not yet on your Drive or if setting up for the first time. If already completed, skip directly to Section 3.

In [ ]:
import json, os
from pathlib import Path

DATASETS = Path("/content/drive/MyDrive/Video-LLaVA/datasets")
annot_file = DATASETS / "annotations/valley_.json"

# 1. Normalize valley_.json paths so they match flat Drive video files
if annot_file.exists():
    with open(annot_file, "r") as f:
        v_data = json.load(f)
    updated = 0
    for item in v_data:
        if "video" in item:
            v_name = Path(item["video"]).name
            new_path = f"valley/{v_name}"
            if item["video"] != new_path:
                item["video"] = new_path
                updated += 1
    if updated > 0:
        with open(annot_file, "w") as f:
            json.dump(v_data, f)
        print(f"\u2705 Normalized {updated:,} paths in valley_.json!")
    else:
        print("\u2705 valley_.json video paths already normalized.")

# 2. Download llava_image.zip to Google Drive if missing
drive_zip = DATASETS / "llava_image.zip"
if not drive_zip.exists() or drive_zip.stat().st_size < 25 * (1024**3):
    print("\nDownloading llava_image.zip (~26.5 GB) with aria2c (~2.5 mins)...")
    !apt-get install -y -qq aria2
    !aria2c -x 16 -s 16 -j 16 -k 1M \
        "https://huggingface.co/datasets/LanguageBind/Video-LLaVA/resolve/main/llava_image.zip" \
        -d /content -o llava_image.zip
    print("Copying archive to Google Drive datasets folder...")
    !cp -f /content/llava_image.zip /content/drive/MyDrive/Video-LLaVA/datasets/llava_image.zip
    print(f"\u2705 Successfully saved to Drive: {drive_zip} ({drive_zip.stat().st_size / (1024**3):.2f} GB)")
else:
    print(f"\u2705 llava_image.zip already verified on Drive ({drive_zip.stat().st_size / (1024**3):.2f} GB).")

## 3. Clone Repository & Install Dependencies
Installs PyTorch dependencies, DeepSpeed ZeRO-2, and optional Flash Attention for A100 acceleration.

In [ ]:
%cd /content

# Clone the repo (skip if already cloned)
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project

# Pull latest updates
!git pull

# Install dependencies (uses Colab's pre-installed PyTorch)
!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm deepspeed huggingface_hub decord gdown av
# Flash Attention accelerates attention computation by 2-3x on A100 (Ampere)
!pip install -q flash-attn --no-build-isolation || echo "Flash attention installation skipped; fallback to standard PyTorch attention."
!apt-get install -y -qq p7zip-full
!pip install -e .

## 4. Option A: Quick Demo / Verification (100 Samples)
Runs a fast dry run on 100 synthetic samples on local SSD in ~2 minutes to verify GPU, forward/backward pass, and DeepSpeed before full training.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --demo_samples 100 \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-demo \
    --num_train_epochs 1.0 \
    --save_steps 25

## 5. Verify Dataset & System Status
Checks disk space and verifies that images (558K) and Valley videos (228K+) are detected on Drive.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action status \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data

## 6. Pretraining Stage 1 (A100 Optimized)

Both commands below:
- Automatically detect the A100 GPU (enabling native **`bf16`**, **`tf32`**, micro-batch size 8 $\times$ 4 gradient accumulation = effective batch size 32).
- On first launch, **automatically unpack `llava_image.zip` to local NVMe SSD in 45 seconds** (`/content/data/llava_image`), feeding the A100 GPU at 2 GB/s without FUSE network latency.
- Stream the 463 GB of Valley videos directly from Google Drive.
- Automatically mule saved checkpoints to Google Drive in the background every 500 steps.

### 6A. Proposal Pretraining (MuLER)
Trains the proposed MuLER architecture and isolates weights into `checkpoints/muler-7b-pretrain/`.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action train \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/muler-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

### 6B. Baseline Pretraining (Optional)
Trains the standard Video-LLaVA baseline from scratch (only needed if an exact local comparison is required). Saves to `checkpoints/videollava-7b-pretrain/`.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action train \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

## 7. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints

## 8. View Secured Checkpoints on Google Drive

In [ ]:
import os
drive_ckpts = "/content/drive/MyDrive/Video-LLaVA/checkpoints"
if os.path.exists(drive_ckpts):
    print("\u2705 Secured Checkpoints on Google Drive:")
    for root, dirs, files in os.walk(drive_ckpts):
        level = root.replace(drive_ckpts, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if f.endswith('.bin') or f.endswith('.json'):
                print(f"{subindent}{f}")
else:
    print("No checkpoints on Drive yet.")